Sentence: "I love AI"

Token IDs:
[10, 25, 42]

GPT-2 Small:
vocab_size = 50,257
context_length = 1024
d_model = 768
num_heads = 12
head_dim = 64

Yes, for **notes**, I'd separate **Training** and **Inference** and keep the `||` style while retaining the important intuition.

---

# TRAINING

### Goal

Input: `I love` || Target: `love AI` || Learn: Given previous tokens, predict the next token.

---

### 1. Tokenization

`"I love AI" → [10,25,42]` || Shape=`(3,)` || Convert text into token IDs.

---

### 2. Token Embedding

`W_E:(50257,768)` || Lookup rows `[10,25,42]` || Output=`(3,768)` || Converts each token ID into a 768-dimensional vector.

---

### 3. Positional Embedding

`W_P:(1024,768)` || Position IDs=`[0,1,2]` || `X = TokenEmb + PosEmb` || Output=`(3,768)` || Injects word order information.

---

### 4. Create Q, K, V

`Wq,Wk,Wv:(768,768)` ||

```text
Q = X @ Wq
K = X @ Wk
V = X @ Wv
```

Output Shapes=`(3,768)`

**Q (Query)** || "What information am I looking for?"
**K (Key)** || "What information do I contain?"
**V (Value)** || "What information can I provide?"

Attention intuition:

```text
Q ↔ K  → decides WHO to attend to
V      → provides the actual information
```

Example:

```text
love(Query)
asks:
Who is performing this action?

I(Key)
advertises:
I am a subject.

I(Value)
contains:
The actual information about "I".
```

---

### 5. Multi-Head Split

12 Heads || `768 → 12 × 64` || Shape=`(3,12,64)` || Each head can learn different relationships (grammar, subject-verb agreement, long-range dependencies, etc.).

---

### 6. Attention Scores

`Q @ Kᵀ` ||

```text
(3,64) @ (64,3)
=
(3,3)
```

Purpose || Every token compares itself with every other token.

Example:

```text
         I   love   AI
love    1.8  2.5   2.2
```

Interpretation:

```text
How relevant is each token to "love"?
```

---

### 7. Scaling

`QKᵀ / √64` || `√64=8`

Purpose:

* Prevent huge dot products
* Prevent overly confident softmax
* Keep gradients stable

---

### 8. Causal Mask

Future tokens must be hidden.

Allowed Attention:

```text
I      → I
love   → I,love
AI     → I,love,AI
```

Mask:

```text
[
 [0,-∞,-∞]
 [0, 0,-∞]
 [0, 0, 0]
]
```

Purpose || Prevent cheating by looking at future tokens.

---

### 9. Softmax

`Softmax(Attention Scores)` || Converts scores into probabilities.

Example:

```text
I      35%
love   65%
AI      0%
```

Properties:

* Values between 0 and 1
* Each row sums to 1

---

### 10. Weighted Sum (Context Creation)

`AttentionWeights @ V`

```text
(3,3) @ (3,64)
=
(3,64)
```
Meaning:

For token: love

attention decided:

  35% information from I

  65% information from love

####So: New_love_vector=0.35*V(I)+0.65*V(love)

This is where context is created.

*Before* : love was just:a standalone word vector

*After* : love contains information about: I too.
_____________________
### 11. Concatenate Heads

Each Head Output=`(3,64)` || 12 Heads ⇒ `(3,768)` || Merge information from all attention heads.

---

### 12. Output Projection

`Wo:(768,768)` || `HeadOutput @ Wo` || Output=`(3,768)` || Mix information from different heads.

---

### 13. Feed Forward Network (FFN)

Architecture:

```text
768 → 3072 → 768
```

Shape Flow:

```text
(3,768)
↓
(3,3072)
↓
(3,768)
```

Attention:

```text
Communication between tokens
```

FFN:

```text
Processing/thinking inside each token
```

---

### 14. Vocabulary Projection

```text
(3,768)
@
(768,50257)
=
(3,50257)
```

Purpose || Produce a score (logit) for every vocabulary token at every position.

Meaning:

```text
For each position:
50,257 possible next-token scores
```

---

### 15. Loss

Targets:

```text
[25,42]
```

Operation:

```text
Cross Entropy Loss
```

Purpose:

* Compare predicted next token vs actual next token
* Measure prediction error
* Lower loss = better prediction

Example:

```text
Position 1:
Predicted → token 30
Actual    → token 25

Error generated
```

---

### 16. Backpropagation

Flow:

```text
Loss
↑
Vocabulary Layer
↑
FFN
↑
Attention
↑
Q,K,V
↑
Embeddings
```

Purpose:

* Compute gradients
* Update weights
* Improve future predictions

Updated Parameters:

```text
W_E
Wq
Wk
Wv
Wo
FFN Weights
```

---

# INFERENCE

### Goal

Input tokens || Predict one new token at a time using already learned weights.

---

### 1. Forward Pass

Same steps as training:

```text
Tokenization
→ Embeddings
→ Position Embeddings
→ Q,K,V
→ Attention
→ FFN
→ Vocabulary Projection
```

No difference up to logits.

---

### 2. Generate Probabilities

```text
Logits
↓
Softmax
↓
Probabilities
```

Example:

```text
AI      60%
ML      25%
Data    10%
Other    5%
```

---

### 3. Choose Next Token

Methods:

* Greedy → highest probability
* Sampling → random based on probabilities
* Top-K
* Top-P (nucleus)

Example:

```text
Choose:
AI
```

---

### 4. Append Token

```text
I love
↓
I love AI
```

New token is added to context.

---

### 5. Repeat

```text
Forward Pass
↓
Generate Next Token
↓
Append Token
↓
Repeat Until EOS/Max Length
```

---

# Training vs Inference

**Training** || Forward Pass → Loss → Backpropagation → Update Weights.

**Inference** || Forward Pass → Select Next Token → Append Token → Repeat.

**Key Difference** || Training learns the weights || Inference uses the learned weights to generate text.


<img src="https://camo.githubusercontent.com/ca9da122cbe41d674059aa95eac087062f9ff9f44d0c0afe02ebe39951b3c81b/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31372e77656270" width =700>

<img src= "https://camo.githubusercontent.com/51962011703851c33b63877f0acd5ca56488e76f80b9b97583168473f65bd0f2/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f31382e77656270" width=700>

| Feature                   | Embedding Layer                      | Linear Layer                                      |                                                  |                       |   |                            |
| ------------------------- | ------------------------------------ | ------------------------------------------------- | ------------------------------------------------ | --------------------- | - | -------------------------- |
| **Primary Use Case**      | Categorical inputs (words, user IDs) | Continuous inputs (hidden states, pixel arrays)   |                                                  |                       |   |                            |
| **Input Format**          | Integer indices (e.g., `3`)          | Dense floating-point vectors (e.g., `[1.2, 0.5]`) |                                                  |                       |   |                            |
| **Operation**             | Dictionary Lookup                    |                                                   | Fetches a pre-allocated vector for a given index | Matrix Multiplication |   | `xWᵀ + b`                  |
| **Memory/Compute**        | Highly efficient                     |                                                   | Avoids multiplying by zeros                      | Computationally dense |   | Scales with large matrices |
| **Weight Representation** | Lookup table matrix `(V × D)`        | Weight matrix `(I × O)` + Bias vector `(O)`       |                                                  |                       |   |                            |




_________________
### Attention Dropout

**Example** || Attention Weights=`[0.1, 0.2, 0.3, 0.4]` || Dropout=`0.5`

**During Training** || Randomly removes some attention connections:

```text
[0.1, 0.2, 0.3, 0.4]
↓
[0.0, 0.2, 0.0, 0.4]
```

|| Remaining weights are automatically rescaled.

**Purpose** || Prevent the model from relying too heavily on a few attention connections.

**Overfitting Example** || Model learns:

```text
journey → always attend to → starts
```

**With Dropout** || Connection may be temporarily removed:

```text
journey ─X─> starts
```

|| Forces the model to learn alternative useful attention patterns.

**Benefit** || Reduces overfitting || Improves generalization and robustness.

<img src="https://camo.githubusercontent.com/a77ef946eb06252af4405c7321883c90a254f4e1647680a754e9c628fae3c369/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830335f636f6d707265737365642f32342e77656270" wodth=600>

In [ ]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CausalSelfAttention(nn.Module):

    def __init__(
        self,
        d_in,              # Input embedding dim (768)
        d_out,             # Head output dim (64)
        context_length,    # Max sequence length
        dropout,
        qkv_bias=False
    ):
        super().__init__()

        self.d_out = d_out

        # Wq : (d_in -> d_out)
        self.W_query = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        # Wk : (d_in -> d_out)
        self.W_key = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        # Wv : (d_in -> d_out)
        self.W_value = nn.Linear(
            d_in,
            d_out,
            bias=qkv_bias
        )

        # Drop attention links during training
        self.dropout = nn.Dropout(dropout)

        # Upper triangular causal mask
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):

        # x : (B,T,d_in)
        B, T, d_in = x.shape

        # (B,T,d_out)
        keys = self.W_key(x)

        # (B,T,d_out)
        queries = self.W_query(x)

        # (B,T,d_out)
        values = self.W_value(x)

        # Similarity matrix
        # (B,T,d_out) @ (B,d_out,T)
        # -> (B,T,T)
        attn_scores = queries @ keys.transpose(1, 2)

        # Hide future tokens
        attn_scores.masked_fill_(
            self.mask.bool()[:T, :T],
            -torch.inf
        )

        # Convert scores -> probabilities
        # (B,T,T)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        # Only active during training
        attn_weights = self.dropout(
            attn_weights
        )

        # Weighted sum of values
        # (B,T,T) @ (B,T,d_out)
        # -> (B,T,d_out)
        context_vec = attn_weights @ values

        return context_vec

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(
        self,
        d_in,
        d_out,
        context_length,
        dropout,
        num_heads,
        qkv_bias=False
    ):
        super().__init__()

        self.heads = nn.ModuleList(

            # num_heads independent heads
            [
                CausalSelfAttention(
                    d_in,
                    d_out,
                    context_length,
                    dropout,
                    qkv_bias
                )

                for _ in range(num_heads)
            ]
        )

        # Wo projection
        self.out_proj = nn.Linear(
            d_out * num_heads,
            d_out * num_heads
        )

    def forward(self, x):

        # Each head:
        # (B,T,d_out)

        # Concatenate:
        # (B,T,d_out*num_heads)
        context_vec = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        # Mix information across heads
        # (B,T,d_out*num_heads)
        context_vec = self.out_proj(
            context_vec
        )

        return context_vec